# Makemore Lesson 03

Live notes and code while following Andrej Karpathy's makemore Part 3.

## Session Goals

- Follow the lesson actively.
- Keep code runnable.
- Add short notes only when a concept is confusing or important.

## 复习前的三个卡住点（来自 Part 2 Session Log）

在看 Part 3 之前，先确认这三个问题清楚了：
1. **滑动窗口**：`context = context[1:] + [ix]`，左边扔掉，右边加新字符
2. **block_size 改动的维度链**：只影响 flatten 后的维度 → 只有 W1 第一个维度变
3. **bigram 不如 MLP**：context 更长 + embedding 泛化能力更强

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

## Data

In [ ]:
words = open('../data/names.txt', 'r').read().splitlines()
words[:8], len(words)

## Notes

- 

## Vocab & Dataset

In [ ]:
# build vocab
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
print(itos)
print('vocab_size:', vocab_size)

In [ ]:
# build dataset
block_size = 3

def build_dataset(words):
  X, Y = [], []
  for w in words:
    context = [0] * block_size
    for ch in w + '.':
      ix = stoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]
  X = torch.tensor(X)
  Y = torch.tensor(Y)
  return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,  Ytr  = build_dataset(words[:n1])   # 80%
Xdev, Ydev = build_dataset(words[n1:n2]) # 10%
Xte,  Yte  = build_dataset(words[n2:])   # 10%
print(Xtr.shape, Xdev.shape, Xte.shape)

## MLP (Part 2 清理版 / Part 3 起点)

In [ ]:
# 超参数用变量名，不再写死数字
n_embd  = 10  # embedding 维度
n_hidden = 200 # hidden layer 大小

g = torch.Generator().manual_seed(2147483647)
C  = torch.randn((vocab_size, n_embd),             generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden),  generator=g)
b1 = torch.randn(n_hidden,                         generator=g)
W2 = torch.randn((n_hidden, vocab_size),            generator=g)
b2 = torch.randn(vocab_size,                        generator=g)
parameters = [C, W1, b1, W2, b2]

print('总参数量:', sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad = True

In [ ]:
# 训练循环
max_steps = 200000
batch_size = 32
lossi = []

for i in range(max_steps):

  # minibatch
  ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
  Xb, Yb = Xtr[ix], Ytr[ix]

  # forward pass
  emb = C[Xb]                        # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1          # hidden layer pre-activation
  h = torch.tanh(hpreact)             # hidden layer
  logits = h @ W2 + b2                # output layer
  loss = F.cross_entropy(logits, Yb)

  # backward pass
  for p in parameters:
    p.grad = None
  loss.backward()

  # update
  lr = 0.1 if i < 100000 else 0.01
  for p in parameters:
    p.data += -lr * p.grad

  # log
  if i % 10000 == 0:
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())

In [ ]:
plt.plot(lossi)

In [ ]:
@torch.no_grad() # 关闭梯度追踪，只做评估
def split_loss(split):
  x, y = {
    'train': (Xtr,  Ytr),
    'val':   (Xdev, Ydev),
    'test':  (Xte,  Yte),
  }[split]
  emb = C[x]                          # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # (N, block_size * n_embd)
  h = torch.tanh(embcat @ W1 + b1)    # (N, n_hidden)
  logits = h @ W2 + b2                 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

## Notes

- 